# 04_03 — Integración de NYC Taxi y NOAA

**Proyecto de Big Data · Fase 04 — Limpieza, transformación e integración de datos**  
**Tecnologías:** Python 3.11 · PySpark 4.1.3 · Parquet · Docker (entorno de pruebas)  
**Periodo de análisis:** 1 de enero – 31 de diciembre de 2024

## Objetivo

Construir un conjunto de datos enriquecido que asocie cada viaje de **NYC Yellow Taxi** con las condiciones meteorológicas diarias observadas por **NOAA** en la estación de Central Park. La integración debe conservar todos los registros de Taxi Silver, garantizar correspondencia temporal y registrar explícitamente los datos meteorológicos faltantes.

| Dataset de entrada | Granularidad | Volumen validado | Función |
|:---|:---|---:|:---|
| `taxi/silver` | Una fila por viaje | **41,169,664** | Tabla principal (*left*) |
| `weather/silver` | Una fila por fecha | **366** | Tabla de referencia (*right*) |

**Criterios de aceptación:** no perder ni duplicar viajes; conservar las 366 fechas; lograr cobertura meteorológica para todas las fechas de inicio; preservar los nulos originales; publicar una salida Parquet validada y particionada por mes.

> **Alcance:** este notebook se dedica a la integración y su persistencia. La limpieza de taxi y NOAA se realizó previamente en `04_01` y `04_02`, respectivamente.


## 1. Configuración del entorno

Se localiza la raíz del repositorio de manera independiente del directorio desde el que se abra Jupyter y se inicializa Spark en modo local.

- `local[4]`: permite utilizar hasta cuatro hilos de ejecución local; se eligió para controlar el consumo de recursos durante el procesamiento.
- `spark.sql.shuffle.partitions=24`: valor inicial para operaciones con intercambio de datos; **no** representa el número de archivos Parquet resultantes.
- `setLogLevel("ERROR")`: reduce los mensajes informativos de Spark en el notebook.

**Importante:** la lectura de datos, los joins y las transformaciones de Spark se evalúan de forma diferida; operaciones como `show`, `count`, `collect` y `write` activan su ejecución.


In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Localizar la raíz del proyecto
# --------------------------------------------------

def find_project_root(start: Path) -> Path:
    """Localiza la raíz del repositorio."""

    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# --------------------------------------------------
# 2. Inicializar Spark
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("04_03_taxi_weather_integration")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "24")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Proyecto:", PROJECT_ROOT)
print("Spark:", spark.version)

Proyecto: c:\Users\derec\Desktop\Proyectos\BigData_project
Spark: 4.1.3


## 2. Lectura de las capas Silver

Se consumen los **datos ya procesados**, no los JSON/Parquet originales. Esto evita repetir la limpieza al ejecutar el notebook de integración y separa claramente las responsabilidades de cada etapa del ETL.

**Entradas utilizadas**

```text
data/processed/
├── taxi/silver/       # Parquet particionado por pickup_month
└── weather/silver/    # Parquet diario de una estación
```

La lectura conserva los tipos definidos en Parquet. En esta etapa verificamos las columnas disponibles; el control de integridad de claves se realiza enseguida.


In [2]:
# --------------------------------------------------
# 1. Definir rutas de entrada
# --------------------------------------------------

TAXI_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "taxi"
    / "silver"
)

WEATHER_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather"
    / "silver"
)

# Verificar que ambas capas existen
assert TAXI_SILVER_PATH.exists(), (
    f"No existe el dataset Taxi Silver: {TAXI_SILVER_PATH}"
)

assert WEATHER_SILVER_PATH.exists(), (
    f"No existe el dataset NOAA Silver: {WEATHER_SILVER_PATH}"
)

# --------------------------------------------------
# 2. Leer archivos Parquet
# --------------------------------------------------

df_taxi = spark.read.parquet(
    str(TAXI_SILVER_PATH)
)

df_weather = spark.read.parquet(
    str(WEATHER_SILVER_PATH)
)

# --------------------------------------------------
# 3. Revisar columnas
# --------------------------------------------------

print("Columnas NYC Taxi:", len(df_taxi.columns))
print("Columnas NOAA:", len(df_weather.columns))

print("\nVariables meteorológicas:")
df_weather.printSchema()

Columnas NYC Taxi: 43
Columnas NOAA: 15

Variables meteorológicas:
root
 |-- weather_date: date (nullable = true)
 |-- awnd_mps: double (nullable = true)
 |-- prcp_mm: double (nullable = true)
 |-- snow_mm: double (nullable = true)
 |-- tmax_c: double (nullable = true)
 |-- tmin_c: double (nullable = true)
 |-- station: string (nullable = true)
 |-- temperature_midpoint_c: double (nullable = true)
 |-- temperature_range_c: double (nullable = true)
 |-- has_precipitation: boolean (nullable = true)
 |-- has_snow: boolean (nullable = true)
 |-- flag_missing_awnd: boolean (nullable = true)
 |-- flag_invalid_temperature: boolean (nullable = true)
 |-- flag_negative_precipitation: boolean (nullable = true)
 |-- flag_negative_snow: boolean (nullable = true)



**Verificación del esquema.** La carga muestra **43 columnas en Taxi Silver** y **15 en NOAA Silver**. La columna `pickup_date` en taxi y `weather_date` en NOAA proporcionan las claves de integración. Las variables meteorológicas incluyen temperatura, precipitación, nieve, viento e indicadores de calidad.

La asociación es temporal y **no geográfica**: todos los viajes iniciados en una misma fecha recibirán la observación de Central Park correspondiente a ese día.


## 3. Validación previa de las claves

La operación que construiremos es un **join muchos-a-uno (N:1)**:

$$
T\;\underset{T.\text{pickup\_date}=W.\text{weather\_date}}{\operatorname{LEFT\ JOIN}}\;W
$$

Donde $T$ representa los viajes y $W$ las observaciones meteorológicas diarias.

La propiedad esencial para evitar multiplicaciones de filas es que **`weather_date` sea única** en la tabla NOAA. También se revisan las columnas requeridas, las fechas nulas y la identificación de la estación. Si alguna comprobación falla, el pipeline debe detenerse antes del join.


In [3]:
# --------------------------------------------------
# 1. Validar columnas necesarias
# --------------------------------------------------

assert "pickup_date" in df_taxi.columns
assert "weather_date" in df_weather.columns

required_weather_columns = [
    "weather_date",
    "station",
    "awnd_mps",
    "prcp_mm",
    "snow_mm",
    "tmax_c",
    "tmin_c",
    "has_precipitation",
    "has_snow",
]

assert all(
    column in df_weather.columns
    for column in required_weather_columns
)

# --------------------------------------------------
# 2. Verificar unicidad de las fechas
# --------------------------------------------------

df_weather_duplicates = (
    df_weather
    .groupBy("weather_date")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_dates = df_weather_duplicates.count()

assert duplicate_dates == 0

# --------------------------------------------------
# 3. Verificar cobertura meteorológica
# --------------------------------------------------

weather_validation = df_weather.agg(
    F.count("*").alias("total_days"),

    F.countDistinct("weather_date").alias(
        "unique_dates"
    ),

    F.sum(
        F.col("weather_date").isNull().cast("long")
    ).alias("null_dates"),

    F.sum(
        F.col("station").isNull().cast("long")
    ).alias("null_stations")
).first()

assert weather_validation["total_days"] == 366
assert weather_validation["unique_dates"] == 366
assert weather_validation["null_dates"] == 0
assert weather_validation["null_stations"] == 0

print("Validaciones de claves correctas.")
print("Fechas meteorológicas:", weather_validation["total_days"])
print("Fechas duplicadas:", duplicate_dates)

Validaciones de claves correctas.
Fechas meteorológicas: 366
Fechas duplicadas: 0


**Resultado:** las **366 fechas meteorológicas son únicas**, no hay claves de fecha nulas ni identificadores de estación nulos. Esta comprobación respalda la cardinalidad N:1 esperada y evita duplicar un viaje por la presencia de varias observaciones NOAA para la misma fecha.


## 4. Integración mediante Broadcast Hash Join

Como la tabla NOAA contiene únicamente 366 filas, se aplica `broadcast(df_weather)` para que Spark disponga de esa tabla pequeña al unirla con los viajes. Es preferible a redistribuir masivamente ambos datasets para esta operación.

Se utiliza `how="left"` por dos motivos:

1. Conservar todos los viajes, incluso si una fecha no tiene registro meteorológico.
2. Diferenciar los problemas de correspondencia temporal de los **valores nulos reales** de una variable meteorológica.

Para ello se agregan dos indicadores:

| Indicador | Significado |
|:---|:---|
| `flag_weather_not_matched` | La fecha del viaje no tuvo fila correspondiente en NOAA. |
| `flag_weather_missing_awnd` | La fecha sí tuvo correspondencia, pero el viento medio diario (`AWND`) es nulo. |

Una vez calculados, `weather_date` se elimina del resultado porque `pickup_date` ya conserva la fecha asociada.


In [4]:
# --------------------------------------------------
# 1. Construir integración
# --------------------------------------------------

df_integrated = (
    df_taxi
    .join(
        F.broadcast(df_weather),

        on=(
            df_taxi["pickup_date"]
            == df_weather["weather_date"]
        ),

        how="left"
    )

    # Identificar viajes sin correspondencia meteorológica
    .withColumn(
        "flag_weather_not_matched",
        F.col("weather_date").isNull()
    )

    # Identificar viento faltante en una fecha
    # que sí tiene registro meteorológico
    .withColumn(
        "flag_weather_missing_awnd",
        F.col("weather_date").isNotNull()
        & F.col("awnd_mps").isNull()
    )

    # Conservar pickup_date como clave principal
    .drop("weather_date")
)

print("Integración definida correctamente.")

# --------------------------------------------------
# 2. Inspeccionar variables integradas
# --------------------------------------------------

df_integrated.select(
    "pickup_date",
    "trip_distance",
    "duration_seconds",
    "prcp_mm",
    "snow_mm",
    "tmax_c",
    "tmin_c",
    "awnd_mps",
    "flag_weather_not_matched"
).show(10, truncate=False)

Integración definida correctamente.
+-----------+-------------+----------------+-------+-------+------+------+--------+------------------------+
|pickup_date|trip_distance|duration_seconds|prcp_mm|snow_mm|tmax_c|tmin_c|awnd_mps|flag_weather_not_matched|
+-----------+-------------+----------------+-------+-------+------+------+--------+------------------------+
|2024-08-24 |0.69         |109             |0.0    |0.0    |28.3  |18.3  |0.6     |false                   |
|2024-08-06 |3.31         |1062            |52.8   |0.0    |32.2  |22.2  |1.5     |false                   |
|2024-08-24 |1.11         |415             |0.0    |0.0    |28.3  |18.3  |0.6     |false                   |
|2024-08-06 |3.02         |1023            |52.8   |0.0    |32.2  |22.2  |1.5     |false                   |
|2024-08-24 |1.67         |466             |0.0    |0.0    |28.3  |18.3  |0.6     |false                   |
|2024-08-06 |3.82         |1247            |52.8   |0.0    |32.2  |22.2  |1.5     |false    

### 4.1 Comprobación del plan físico

La llamada `explain(mode="formatted")` permite inspeccionar cómo Spark planifica la unión. Buscamos las operaciones **`BroadcastExchange`**, **`BroadcastHashJoin LeftOuter BuildRight`** y **`AdaptiveSparkPlan`**.

Esta inspección documenta la estrategia utilizada, más allá de comprobar únicamente que el código se ejecuta sin errores.


In [5]:
# Inspeccionar el plan físico de ejecución
df_integrated.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- BroadcastHashJoin LeftOuter BuildRight (5)
      :- Scan parquet  (1)
      +- BroadcastExchange (4)
         +- Filter (3)
            +- Scan parquet  (2)


(1) Scan parquet 
Output [43]: [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, source_month#19, pickup_date#20, pickup_hour#21, pickup_dayofweek#22, is_weekend#23, duration_seconds#24L, is_flex_fare#25, flag_out_of_year#26, flag_month_mismatch#27, flag_invalid_duration#28, flag_zero_distance#29, flag_missing_or_negative_distance#30, flag_missing_location#31, flag_nonpositive_location_id#32, is_eligible_for_daily_volume#33, is_candidate_mobility#34, speed_mph#35, flag_speed_over_80#3

**Lectura del plan observado.** Spark confirma `BroadcastHashJoin LeftOuter BuildRight`: la tabla NOAA se utiliza como el lado pequeño para construir el hash y el lado izquierdo (Taxi) se conserva. `BroadcastExchange` corresponde a la distribución de NOAA y `AdaptiveSparkPlan` indica que la optimización adaptativa está habilitada.

El resultado proyecta **59 columnas**: 43 de taxi, 14 de NOAA después de retirar `weather_date` y 2 indicadores de integración. El plan detallado se conserva en la salida de la celda anterior; puede desplegarse si se requiere auditarlo.


## 5. Validación funcional de la integración

Se ejecuta una agregación sobre el DataFrame enriquecido para verificar los criterios de aceptación. Al tratarse de un `LEFT JOIN` con fechas únicas en NOAA, el número de viajes debe permanecer inalterado.

Se comprueba el total de filas, la correspondencia de fechas y los viajes asociados a los días en que no existe una observación de viento. Esta última cifra **no es una tasa de error del join**, sino una propagación de valores faltantes de la fuente meteorológica.


In [6]:
# --------------------------------------------------
# 1. Resumen del JOIN
# --------------------------------------------------

df_join_validation = df_integrated.agg(

    F.count("*").alias("total_integrated"),

    # Viajes sin correspondencia meteorológica
    F.sum(
        F.col("flag_weather_not_matched").cast("long")
    ).alias("unmatched_weather"),

    # Viajes correspondientes a fechas con AWND faltante
    F.sum(
        F.col("flag_weather_missing_awnd").cast("long")
    ).alias("missing_awnd_trips"),

    # Fechas distintas en el dataset integrado
    F.countDistinct("pickup_date").alias(
        "unique_pickup_dates"
    )
)

df_join_validation.show(truncate=False)

# --------------------------------------------------
# 2. Validaciones automáticas
# --------------------------------------------------

result = df_join_validation.first()

assert result["total_integrated"] == 41_169_664, (
    "La integración modificó el número esperado de viajes."
)

assert result["unmatched_weather"] == 0, (
    "Existen viajes sin correspondencia meteorológica."
)

assert result["unique_pickup_dates"] == 366, (
    "No se conservaron todas las fechas de 2024."
)

print("Validaciones iniciales del JOIN correctas.")

+----------------+-----------------+------------------+-------------------+
|total_integrated|unmatched_weather|missing_awnd_trips|unique_pickup_dates|
+----------------+-----------------+------------------+-------------------+
|41169664        |0                |428126            |366                |
+----------------+-----------------+------------------+-------------------+

Validaciones iniciales del JOIN correctas.


### 5.1 Resultados

| Control | Resultado observado | Interpretación |
|:---|---:|:---|
| Viajes integrados | **41,169,664** | Coincide con Taxi Silver. |
| Viajes sin correspondencia meteorológica | **0** | Todos encontraron una fecha NOAA. |
| Fechas distintas de inicio | **366** | Cobertura temporal completa de 2024. |
| Viajes con `AWND` faltante | **428,126** | **1.04 %** de los viajes; fuente NOAA, no fallo del join. |

Los valores faltantes de `AWND` proceden de **cuatro fechas** identificadas previamente en `04_02` (24 y 27 de enero; 23 y 24 de junio). Todos los viajes ocurridos en esas fechas heredan ese valor nulo, por lo que es normal que la cantidad de viajes afectados sea considerablemente mayor que cuatro.

> **Decisión:** conservar los viajes y sus nulos meteorológicos. No se imputan valores de viento durante la integración.


## 6. Persistencia incremental por mes

Una vez verificada la integración en memoria, se ejecuta el módulo reutilizable `src/etl/taxi_weather_integration.py` sobre cada partición mensual de Taxi Silver.

**Flujo de persistencia**

```text
Taxi Silver (pickup_month=AAAA-MM) ──┐
                                     ├─> Broadcast LEFT JOIN ─> _silver_staging/
NOAA Silver (366 días) ──────────────┘                      (Parquet/Snappy)
                                                               │
                                               Validación de integridad
                                                               │
                                                        Promoción a silver/
```

La escritura utiliza `pickup_month` —el mes real del viaje— y no `source_month` —el mes del archivo original—, ya que ambos pueden diferir en algunos registros. Las doce particiones mensuales se escriben con compresión **Snappy**, utilizando `maxRecordsPerFile=500_000` como límite de registros por archivo.

**Precaución de reejecución:** el directorio `_silver_staging` debe **no existir** al iniciar la escritura. Los modos `append` y `errorifexists` evitan una repetición silenciosa sobre el mismo staging; no deben usarse para reanudar a ciegas una ejecución parcial. Como el dataset final de este notebook ya fue publicado, **no reejecutar esta celda** salvo que se esté realizando una nueva corrida controlada en una ruta nueva.


In [7]:
import sys
import importlib

from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Importar módulo reutilizable
# --------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.etl.taxi_weather_integration as integration

importlib.reload(integration)

# --------------------------------------------------
# 2. Definir rutas
# --------------------------------------------------

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "taxi_weather"
)

STAGING_PATH = OUTPUT_ROOT / "_silver_staging"

FINAL_PATH = OUTPUT_ROOT / "silver"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Evitar duplicaciones por reejecución.
if STAGING_PATH.exists():
    raise FileExistsError(
        f"Ya existe el directorio temporal: {STAGING_PATH}. "
        "Revísalo antes de iniciar una nueva escritura."
    )

# --------------------------------------------------
# 3. Procesamiento incremental
# --------------------------------------------------

for month in range(1, 13):

    month_label = f"2024-{month:02d}"

    print(f"Procesando {month_label}...", flush=True)

    # Leer Taxi Silver.
    # El filtro sobre pickup_month permite a Spark
    # limitar la lectura a la partición mensual.
    df_month = (
        spark.read.parquet(str(TAXI_SILVER_PATH))
        .filter(
            F.col("pickup_month") == month_label
        )
    )

    # Integrar exclusivamente los viajes del mes
    # con la tabla meteorológica.
    df_month_integrated = (
        integration.integrate_taxi_weather(
            df_month,
            df_weather
        )
    )

    # Primera escritura: crear dataset.
    # Meses siguientes: agregar nuevos archivos.
    write_mode = (
        "errorifexists" if month == 1 else "append"
    )

    (
        df_month_integrated
        .write
        .mode(write_mode)
        .option("compression", "snappy")
        .option("maxRecordsPerFile", 500_000)
        .partitionBy("pickup_month")
        .parquet(str(STAGING_PATH))
    )

    print(
        f"Mes {month_label} almacenado correctamente.",
        flush=True
    )

print("Integración incremental completada.")

Procesando 2024-01...
Mes 2024-01 almacenado correctamente.
Procesando 2024-02...
Mes 2024-02 almacenado correctamente.
Procesando 2024-03...
Mes 2024-03 almacenado correctamente.
Procesando 2024-04...
Mes 2024-04 almacenado correctamente.
Procesando 2024-05...
Mes 2024-05 almacenado correctamente.
Procesando 2024-06...
Mes 2024-06 almacenado correctamente.
Procesando 2024-07...
Mes 2024-07 almacenado correctamente.
Procesando 2024-08...
Mes 2024-08 almacenado correctamente.
Procesando 2024-09...
Mes 2024-09 almacenado correctamente.
Procesando 2024-10...
Mes 2024-10 almacenado correctamente.
Procesando 2024-11...
Mes 2024-11 almacenado correctamente.
Procesando 2024-12...
Mes 2024-12 almacenado correctamente.
Integración incremental completada.


**Resultado de la ejecución:** se procesaron satisfactoriamente los doce meses de **enero a diciembre de 2024**. El staging quedó disponible para comprobar el volumen, la cobertura y la consistencia mensual antes de publicarlo.


## 7. Validación de la persistencia

Se leen los archivos generados desde el directorio temporal para verificar la **salida física**, no únicamente el DataFrame lógico del join.

Los controles comprueban que:

- Persistan las **41,169,664 filas** y las **366 fechas**.
- No existan viajes sin correspondencia NOAA.
- Los **428,126 viajes con `AWND` faltante** sigan correctamente identificados.
- Los conteos de viajes de cada `pickup_month` coincidan exactamente entre Taxi Silver y el resultado guardado.

La comparación por mes proporciona una validación más específica que comprobar solamente el número total de registros.


In [8]:
# --------------------------------------------------
# 1. Leer el dataset integrado almacenado
# --------------------------------------------------

df_saved = spark.read.parquet(
    str(STAGING_PATH)
)

# --------------------------------------------------
# 2. Validar volumen y cobertura meteorológica
# --------------------------------------------------

validation = df_saved.agg(

    F.count("*").alias("total_records"),

    F.countDistinct("pickup_date").alias(
        "unique_dates"
    ),

    F.sum(
        F.col("flag_weather_not_matched").cast("long")
    ).alias("unmatched_weather"),

    F.sum(
        F.col("flag_weather_missing_awnd").cast("long")
    ).alias("missing_awnd_trips")
).first()

print("Registros:", validation["total_records"])
print("Fechas:", validation["unique_dates"])
print("Sin correspondencia:", validation["unmatched_weather"])
print("Viajes sin AWND:", validation["missing_awnd_trips"])

assert validation["total_records"] == 41_169_664
assert validation["unique_dates"] == 366
assert validation["unmatched_weather"] == 0
assert validation["missing_awnd_trips"] == 428_126

# --------------------------------------------------
# 3. Comparar distribución mensual
# --------------------------------------------------

def monthly_counts(df):
    """Devuelve un diccionario {mes: cantidad}."""

    rows = (
        df.groupBy("pickup_month")
        .count()
        .collect()
    )

    return {
        row["pickup_month"]: row["count"]
        for row in rows
    }


taxi_monthly = monthly_counts(df_taxi)
integrated_monthly = monthly_counts(df_saved)

assert taxi_monthly == integrated_monthly
assert len(integrated_monthly) == 12

print("Distribución mensual conservada.")
print("Todas las validaciones de integración pasaron.")

Registros: 41169664
Fechas: 366
Sin correspondencia: 0
Viajes sin AWND: 428126
Distribución mensual conservada.
Todas las validaciones de integración pasaron.


**Resultado:** el volumen global y la cobertura meteorológica se conservaron, y los **12 conteos mensuales coinciden** con el dataset Taxi Silver. No se detectaron pérdidas o duplicaciones a nivel de conteos durante la persistencia incremental.

Esta validación corresponde a las reglas explícitas del pipeline; no implica que todas las observaciones originales de viajes o las medidas meteorológicas estén libres de errores.


## 8. Publicación del dataset definitivo

La salida validada se mueve del directorio `_silver_staging` al destino de publicación `data/processed/taxi_weather/silver/`.

La operación evita sobrescribir una ruta definitiva existente y confirma que el conjunto publicado puede leerse correctamente. Esta promoción de directorios es adecuada para el flujo local del proyecto, pero **no constituye una transacción ACID** ni reemplaza los controles de escritura de un formato de tabla como Delta Lake.

> **Nota de reejecución:** esta celda ya se ejecutó exitosamente. Después de la promoción, `STAGING_PATH` deja de existir y `FINAL_PATH` sí existe; por ello, volver a ejecutarla generará una excepción **intencional**. Esto protege el artefacto ya publicado.


In [10]:
# La promoción no sobrescribe resultados existentes.
if FINAL_PATH.exists():
    raise FileExistsError(
        f"El destino definitivo ya existe: {FINAL_PATH}"
    )

if not STAGING_PATH.exists():
    raise FileNotFoundError(
        f"No existe el staging: {STAGING_PATH}"
    )

# Ambas rutas están bajo el mismo directorio padre.
STAGING_PATH.rename(FINAL_PATH)

# Verificar que el destino definitivo puede leerse.
df_final = spark.read.parquet(str(FINAL_PATH))

assert df_final.count() == 41_169_664

print("Dataset integrado publicado correctamente.")
print("Ruta definitiva:", FINAL_PATH)

Dataset integrado publicado correctamente.
Ruta definitiva: c:\Users\derec\Desktop\Proyectos\BigData_project\data\processed\taxi_weather\silver


## 9. Conclusiones y consideraciones

### Resultados alcanzados

- Se integraron **41,169,664 viajes** de NYC Taxi Silver con las **366 fechas** de NOAA Silver.
- Se confirmó un **Broadcast Hash Join** con preservación del lado de taxi y sin correspondencias meteorológicas perdidas.
- Se identificaron **428,126 viajes (1.04 %)** afectados por cuatro fechas con viento faltante; se conservaron sin imputación.
- La escritura incremental por `pickup_month` permitió almacenar los doce meses en Parquet/Snappy y validar los conteos de cada partición.
- Se publicó correctamente la capa enriquecida en `data/processed/taxi_weather/silver/`.

### Limitaciones metodológicas

1. **Representatividad espacial:** el clima proviene de una única estación de Central Park; no describe necesariamente las condiciones meteorológicas de cada zona de recogida.
2. **Resolución temporal:** las medidas de NOAA son diarias. El join no asigna el clima real de la **hora** exacta de cada viaje.
3. **Predicción temporal:** variables calculadas con el día completo no deben utilizarse como predictores conocidos al inicio de ese mismo día sin verificar su disponibilidad, para evitar **fuga de información**.
4. **Calidad de origen:** la integración preserva los indicadores de calidad de Taxi y NOAA; no sustituye validaciones adicionales de negocio.

### Continuidad del proyecto

El dataset resultante queda listo para **`04_04 — Validación integral del pipeline`**, donde se podrán consolidar controles de esquema, integridad, cobertura y reproducibilidad de extremo a extremo.

**Nota de reproducción:** el notebook conserva las salidas reales de la corrida validada. Las secciones de escritura y publicación son de ejecución única; si se desea repetir el ETL, debe prepararse un destino nuevo y ejecutar el proceso completo de manera controlada.
